In [225]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [205]:
# Material Parameters
mu = 0.8
jm = 40.0
Identity = torch.eye(3)

In [206]:
# Synthetic Data Generation
def generate_synthetic_data(num_samples, lamda_min, lamda_max,eta_min,eta_max):
    lamda_train = torch.linspace(lamda_min, lamda_max, num_samples)
    I1_train = lamda_train**2 + 2/lamda_train
    P_true_train = gent_stress_function(lamda_train, mu, jm)   
    noise = add_noise(P_true_train,eta_min,eta_max,lamda_train)
    Y_train = P_true_train + noise
    Y_train = Y_train.reshape(-1, 1)
    lamda_train =lamda_train.reshape(-1, 1)
    I1_train = I1_train.reshape(-1, 1)
    x = torch.cat((lamda_train, I1_train,Y_train), dim=1)
    return x


In [207]:
# Gent Stress Function
def gent_stress_function(lamda, mu, jm):
    I1 = lamda**2 + 2/lamda
    stress = mu * (lamda - 1/lamda**2) / (1 - (I1 - 3)/jm)
    return stress

In [208]:
def deformation_gradient(lamda):
    diagonal_values = torch.stack([
        lamda,
        lamda**(-0.5),
        lamda**(-0.5)
    ], dim=-1)

    return torch.diag_embed(diagonal_values)

In [209]:
def frobenius_norm(F):
    return torch.sqrt(
        torch.sum((F - Identity)**2, dim=(-2, -1))
    )

In [227]:
def add_noise(P_true_train,eta_min,eta_max,lamda):
    P_char = torch.max(torch.abs(P_true_train))
    noise_sd_min = eta_min * P_char
    noise_sd_max = eta_max * P_char
    F = deformation_gradient(lamda)
    F_lambdamax = deformation_gradient(torch.max(lamda))
    q = 2.0
    t = frobenius_norm(F)
    ksy = torch.randn_like(P_true_train)
    conditional_noise_variance = torch.square(noise_sd_min) + (torch.square(noise_sd_max) - torch.square(noise_sd_min)) * ((t / frobenius_norm(F_lambdamax))**q)
    return torch.sqrt(conditional_noise_variance) * ksy

In [226]:
x = generate_synthetic_data(num_samples=50, lamda_min=1.0, lamda_max=4.0, eta_min=0.005,eta_max=0.03)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self,m = 10):
        super().__init__()
        self.a = nn.Parameter(torch.randn(1))
        self.c = nn.Parameter(torch.randn(m))
        self.w = nn.Parameter(torch.randn(m))
        self.b = nn.Parameter(torch.randn(m))
    
    def forward(self,I1):
        a = F.softplus(self.a)
        c = F.softplus(self.c)
        w = F.softplus(self.w)
        x = I1 - 3
        z = x * w + self.b

        # Broadcasting (element wise multiplication)

        hidden = (F.softplus(z) - F.softplus(self.b))
        psi = (a * x + torch.sum(c * hidden, dim = -1, keepdim = True))
        return psi

In [213]:
model = NeuralNetwork()


In [219]:
energy_potential = model(x[:,-1].reshape(-1,1))

In [222]:
x.shape

torch.Size([50, 3])

In [224]:
energy_potential.shape

torch.Size([50, 1])